# 5. Ajuste fino de hiperparámetros

`GridSearchCV` con **validación cruzada temporal** sobre el entrenamiento (criterio: MSE),
reentrenamiento del mejor con todo el entrenamiento y evaluación **una sola vez** en la prueba
(capítulo 9.2 del curso). Las mallas están en la celda de configuración: se pueden ampliar o reducir.
En P3, Prophet ajusta `changepoint_prior_scale` en un periodo de validación previo a la prueba y VAR
elige rezagos por AIC; los modelos de ML se reajustan en cada origen con su propia búsqueda.

Tiempo aproximado en una RTX 4050 con las mallas por defecto: P1 ~10 min, P2 ~2 h, P3 ~1 h.

In [ ]:
import sys, warnings, json, time
from pathlib import Path
ML = Path.cwd().parent                      # carpeta ml/
sys.path.insert(0, str(ML))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
PROCESSED, REPORTS = ML / "data" / "processed", ML / "reports"
PRED = REPORTS / "predicciones"; PRED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", palette="deep")

In [ ]:
# ===================== CONFIGURACIÓN =====================
USAR_GPU = True
CORRER = {"p1": True, "p2": True, "p3": True}
P3_HORIZONTES = [1, 3, 6, 12]
PROPHET_CPS = [0.01, 0.05, 0.2, 0.5]     # malla de changepoint_prior_scale
VAR_MAXLAGS = 6

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from mirage.ml import gpu as G
CFG = json.load(open(PROCESSED / "configuracion_particion.json"))
DEV = "cuda" if USAR_GPU else "cpu"
print("Dispositivo de los modelos:", DEV, "| GPU disponible:", G.device_disponible())

def preprocesador(num, cat):
    numerico = Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())])
    categorico = Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                           ("codificar", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False))])
    return ColumnTransformer([("num", numerico, num)] + ([("cat", categorico, cat)] if cat else []))

def pipeline(est, num, cat):
    return Pipeline([("prep", preprocesador(num, cat)), ("modelo", est)])

def pliegues_temporales(periodos, n_splits=CFG["N_PLIEGUES_CV"]):
    u = np.sort(pd.Series(periodos).unique())
    return [(np.where(pd.Series(periodos).isin(u[a]))[0], np.where(pd.Series(periodos).isin(u[b]))[0])
            for a, b in TimeSeriesSplit(n_splits=min(n_splits, len(u) - 1)).split(u)]

def metricas(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float); nz = np.abs(y) > 1e-9
    return {"R2": r2_score(y, p), "MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "MAPE_%": 100 * mean_absolute_percentage_error(y[nz], p[nz])}

def cargar(problema):
    """X, Y, periodo, columnas y máscara de entrenamiento de P1 o P2, con la partición guardada."""
    d = pd.read_parquet(PROCESSED / f"dataset_{problema}.parquet")
    d = d.merge(pd.read_parquet(PROCESSED / f"particion_{problema}.parquet"), on="id")
    if problema == "p1":
        Y = d[[c for c in d.columns if c.startswith("y_")]]; periodo = d["mes"]
        X = d.drop(columns=list(Y.columns) + ["split", "id", "mes"])
    else:
        Y = d[["y_de", "y_cob", "y_kd", "y_roa"]]; periodo = d["anio_objetivo"]
        X = d[["ciiu_division", "log_activos", "de", "de_l1", "cob", "kd", "roa", "roa_l1", "margen", "caja_activos",
               "deuda_activos", "crec_deuda", "tasa_impuesto_efectiva", "tasa_politica", "tes_10a_banrep", "ust_10a"]]
    cat = ["ciiu_division"]; num = [c for c in X.columns if c not in cat]
    return X, Y, periodo, num, cat, (d["split"] == "train").to_numpy(), d["id"].to_numpy()

def persistencia(problema, X, Y):
    fuente = {"p1": {c: c.replace("y_", "") + "_l0" for c in Y.columns},
              "p2": {"y_de": "de", "y_cob": "cob", "y_kd": "kd", "y_roa": "roa"}}[problema]
    return pd.DataFrame({c: X[s].to_numpy() for c, s in fuente.items()}).fillna(Y.median())

In [ ]:
# ===================== MODELOS Y MALLAS DE BÚSQUEDA =====================
# Los parámetros llevan el prefijo "modelo__" (nombre del paso en el pipeline).
MODELOS = {
    "Ridge":              (G.GPULinear(device=DEV), DEV),
    "Lasso":              (G.GPULasso(device=DEV), DEV),
    "k-NN":               (G.GPUKNN(device=DEV), DEV),
    "Árbol de decisión":  (G.xgb_tree(device=DEV), DEV),
    "Random Forest":      (G.xgb_forest(device=DEV), DEV),
    "Gradient Boosting":  (G.xgb_boost(learning_rate=0.1, device=DEV), DEV),
    "XGBoost":            (G.xgb_boost(device=DEV), DEV),
    "SVR":                (G.GPUSVR(device=DEV), DEV),
    "Red neuronal (MLP)": (G.GPUMLP(device=DEV), DEV),
}
MALLAS = {
    "Ridge":             {"modelo__alpha": [0.01, 0.1, 1, 10, 100]},
    "Lasso":             {"modelo__alpha": [1e-4, 1e-3, 1e-2, 1e-1]},
    "k-NN":              {"modelo__n_neighbors": [5, 15, 30, 60], "modelo__weights": ["uniform", "distance"]},
    "Árbol de decisión": {"modelo__max_depth": [4, 8, 12], "modelo__min_child_weight": [20, 100]},
    "Random Forest":     {"modelo__n_estimators": [100, 300], "modelo__max_depth": [8, 12, 16],
                          "modelo__subsample": [0.6], "modelo__colsample_bynode": [0.6]},
    "Gradient Boosting": {"modelo__n_estimators": [200, 500], "modelo__max_depth": [3, 6], "modelo__learning_rate": [0.03, 0.1]},
    "XGBoost":           {"modelo__n_estimators": [200, 500], "modelo__max_depth": [3, 6], "modelo__learning_rate": [0.03, 0.1],
                          "modelo__subsample": [0.8, 1.0], "modelo__reg_lambda": [1, 10]},
    "SVR":               {"modelo__C": [0.1, 1, 10], "modelo__epsilon": [0.01, 0.1], "modelo__n_components": [2000]},
    "Red neuronal (MLP)": {"modelo__hidden_layer_sizes": [(32,), (64, 32), (128, 64)], "modelo__alpha": [1e-4, 1e-2],
                           "modelo__learning_rate_init": [1e-3, 1e-2]},
}
# P3 tiene 127 meses y reentrena en cada uno de los 36 meses de prueba × 4 horizontes:
# mallas pequeñas para que la búsqueda sea viable (cada combinación se ajusta 36 × 4 × 3 veces).
MALLAS_P3 = {
    "Ridge":         {"modelo__alpha": [0.1, 1, 10, 100]},
    "Random Forest": {"modelo__n_estimators": [100], "modelo__max_depth": [4, 8]},
    "XGBoost":       {"modelo__n_estimators": [200], "modelo__max_depth": [2, 4], "modelo__learning_rate": [0.03, 0.1]},
}
pd.DataFrame({"modelo": list(MALLAS), "combinaciones": [int(np.prod([len(v) for v in m.values()])) for m in MALLAS.values()],
              "dispositivo": [MODELOS[k][1] for k in MALLAS]})

In [ ]:
def correr(problema, modelos, etapa, mallas=None):
    X, Y, periodo, num, cat, tr, ids = cargar(problema)
    filas = []
    for nombre, (est, dev) in modelos.items():
        t0 = time.time(); pipe = pipeline(clone(est), num, cat); info = {}
        malla = (mallas or {}).get(nombre)
        if malla:
            gs = GridSearchCV(pipe, malla, cv=pliegues_temporales(periodo[tr].reset_index(drop=True)),
                              scoring="neg_mean_squared_error", n_jobs=1, refit=True)
            gs.fit(X[tr].reset_index(drop=True), Y[tr].reset_index(drop=True)); modelo = gs.best_estimator_
            info = {"mejores_parametros": {k: str(v) for k, v in gs.best_params_.items()}, "mse_cv": -gs.best_score_}
        else:
            modelo = pipe.fit(X[tr], Y[tr])
        seg = time.time() - t0
        P_tr = pd.DataFrame(np.asarray(modelo.predict(X[tr])).reshape(tr.sum(), -1), columns=Y.columns)
        P_te = pd.DataFrame(np.asarray(modelo.predict(X[~tr])).reshape((~tr).sum(), -1), columns=Y.columns)
        P_te.assign(id=ids[~tr]).to_parquet(PRED / f"{problema}_{etapa}_{nombre}.parquet", index=False)
        for c in Y.columns:
            filas.append({"modelo": nombre, "objetivo": c, "dispositivo": dev, "segundos": seg,
                          "R2_entrenamiento": r2_score(Y.loc[tr, c], P_tr[c]), **metricas(Y.loc[~tr, c], P_te[c]),
                          "parametros": json.dumps(info.get("mejores_parametros", {}), ensure_ascii=False)})
        print(f"{problema} · {nombre:20s} [{dev}] {seg:7.1f} s", flush=True)
    pers = persistencia(problema, X[~tr].reset_index(drop=True), Y[tr])
    pers.assign(id=ids[~tr]).to_parquet(PRED / f"{problema}_{etapa}_Persistencia.parquet", index=False)
    for c in Y.columns:
        filas.append({"modelo": "Persistencia", "objetivo": c, "dispositivo": "-", "segundos": 0, "R2_entrenamiento": np.nan,
                      **metricas(Y.loc[~tr, c], pers[c]), "parametros": "{}"})
    Y[~tr].assign(id=ids[~tr]).to_parquet(PRED / f"{problema}_real.parquet", index=False)
    res = pd.DataFrame(filas); res.to_csv(REPORTS / f"{problema}_{etapa}_metricas.csv", index=False)
    return res

def resumen(res):
    # Tolera tablas de corridas anteriores sin todas las columnas
    for c, v in (("dispositivo", "cpu"), ("segundos", np.nan), ("R2_entrenamiento", np.nan)):
        if c not in res: res[c] = v
    if "conjunto" in res:   # formato de la corrida con scripts: separar entrenamiento y prueba
        tr = res[res.conjunto == "entrenamiento"].set_index(["modelo", "objetivo"])["R2"]
        res = res[res.conjunto == "prueba"].copy(); res["R2_entrenamiento"] = res.set_index(["modelo", "objetivo"]).index.map(tr)
    t = res.groupby("modelo").agg(dispositivo=("dispositivo", "first"), segundos=("segundos", "first"), R2_entrenamiento=("R2_entrenamiento", "mean"),
                                  R2=("R2", "mean"), MAE=("MAE", "mean"), RMSE=("RMSE", "mean"))
    t["brecha_%"] = 100 * (t.R2_entrenamiento - t.R2) / t.R2_entrenamiento.abs()
    t["diagnostico"] = np.select([t["brecha_%"] < 10, t["brecha_%"] < 15], ["aceptable", "revisar"], "sobreajuste")
    return t.sort_values("RMSE").round(3)

## 5.1 P1 · Kd por sector y tamaño

In [ ]:
if CORRER["p1"]: a1 = correr("p1", MODELOS, "ajuste_fino", MALLAS)
a1 = pd.read_csv(REPORTS / "p1_ajuste_fino_metricas.csv"); resumen(a1)

## 5.2 P2 · Estructura de capital por empresa

In [ ]:
if CORRER["p2"]: a2 = correr("p2", MODELOS, "ajuste_fino", MALLAS)
a2 = pd.read_csv(REPORTS / "p2_ajuste_fino_metricas.csv"); resumen(a2)

## 5.3 P3 · Insumos de mercado

In [ ]:
P3 = pd.read_parquet(PROCESSED / "dataset_p3.parquet")
OBJ3 = ["spread_soberano", "tes_cop_10a", "tes_uvr_10a", "inflacion_implicita_10a", "tasa_politica", "ust_10a"]

In [ ]:
# ---------- Modelos de series de tiempo (CPU: no tienen versión en GPU) ----------
import logging; logging.getLogger("cmdstanpy").setLevel(logging.ERROR); logging.getLogger("prophet").setLevel(logging.ERROR)
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.api import VAR
from prophet import Prophet

def paseo_aleatorio(train, h, **kw): return train.iloc[-1]

def arima(train, h, ordenes=((1, 1, 0), (0, 1, 1), (1, 1, 1), (2, 1, 0)), **kw):
    out = {}
    for c in train.columns:
        mejor, aic = None, np.inf
        for o in ordenes:
            try:
                r = ARIMA(train[c].values, order=o).fit()
                if r.aic < aic: mejor, aic = r, r.aic
            except Exception: pass
        out[c] = mejor.forecast(h)[-1] if mejor is not None else train[c].iloc[-1]
    return pd.Series(out)

def var(train, h, maxlags=2, **kw):
    d = train.diff().dropna()
    try:
        r = VAR(d).fit(maxlags=maxlags, ic="aic"); k = max(r.k_ar, 1)
        return train.iloc[-1] + r.forecast(d.values[-k:], steps=h).sum(axis=0)
    except Exception:
        return train.iloc[-1]

def prophet(train, h, cps=0.05, **kw):
    out = {}
    for c in train.columns:
        m = Prophet(changepoint_prior_scale=cps, yearly_seasonality=True, weekly_seasonality=False,
                    daily_seasonality=False, uncertainty_samples=0).fit(pd.DataFrame({"ds": train.index, "y": train[c].values}))
        out[c] = float(m.predict(pd.DataFrame({"ds": [train.index[-1] + pd.offsets.MonthEnd(h)]}))["yhat"].iloc[0])
    return pd.Series(out)

def origen_movil(S, modelo, h, meses_prueba, minimo=48, **kw):
    """En cada mes t del periodo de prueba se reentrena con la historia hasta t y se pronostica t+h."""
    idx, filas = S.index, {}
    for t in idx[-(meses_prueba + h):-h]:
        tr = S.loc[:t]
        if len(tr) >= minimo: filas[idx[idx.get_loc(t) + h]] = modelo(tr, h, **kw)
    return pd.DataFrame(filas).T.sort_index()

def supervisado_p3(P3, h, rezagos=3):
    """Estrategia directa: con la información hasta t, predecir el vector de objetivos en t+h."""
    f = {}
    for c in OBJ3 + ["ibr_3m", "bei_usd_10a"]:
        for k in range(rezagos): f[f"{c}_l{k}"] = P3[c].shift(k)
        f[f"{c}_d1"] = P3[c].diff()
    f["colcap_ret"] = P3["colcap"].pct_change(fill_method=None); f["trm_ret"] = P3["trm"].pct_change(fill_method=None)
    X = pd.DataFrame(f, index=P3.index); Y = P3[OBJ3].shift(-h); ok = X.notna().all(1) & Y.notna().all(1)
    return X[ok], Y[ok]

def ml_origen_movil(P3, h, estimador, malla=None, meses_prueba=36):
    """Modelos de ML con rezagos, reentrenados en cada origen (y con GridSearchCV si se pasa malla)."""
    X, Y = supervisado_p3(P3, h); S = P3[OBJ3]; preds = {}
    for t in Y.index[-meses_prueba:]:
        pos = Y.index.get_loc(t); tr = Y.index[:max(pos - h + 1, 0)]   # solo filas cuyo objetivo ya se conoce en t
        if len(tr) < 36: continue
        pipe = pipeline(clone(estimador), list(X.columns), [])
        if malla: pipe = GridSearchCV(pipe, malla, cv=TimeSeriesSplit(n_splits=3), scoring="neg_mean_squared_error")
        pipe.fit(X.loc[tr], Y.loc[tr]); j = S.index.get_loc(t) + h
        if j < len(S): preds[S.index[j]] = pd.Series(np.ravel(pipe.predict(X.loc[[t]]))[:len(OBJ3)], index=OBJ3)
    return pd.DataFrame(preds).T.sort_index()

In [ ]:
def correr_p3_ajuste():
    S, filas = P3[OBJ3], []
    val = S.iloc[:-CFG["P3_MESES_PRUEBA"]]          # validación: antes del periodo de prueba
    for h in P3_HORIZONTES:
        err = {c: ((origen_movil(val, prophet, h, 12, cps=c) - val) ** 2).mean().mean() for c in PROPHET_CPS}
        cps = min(err, key=err.get); print(f"h={h}: Prophet cps elegido {cps}")
        series = {"Paseo aleatorio": (paseo_aleatorio, {}), "ARIMA": (arima, {}), "VAR": (var, {"maxlags": VAR_MAXLAGS}),
                  "Prophet": (prophet, {"cps": cps})}
        for nombre, (fn, kw) in series.items():
            t0 = time.time(); f = origen_movil(S, fn, h, CFG["P3_MESES_PRUEBA"], **kw); seg = time.time() - t0
            f.to_parquet(PRED / f"p3_ajuste_fino_h{h}_{nombre}.parquet")
            for c in OBJ3: filas.append({"modelo": nombre, "h": h, "objetivo": c, "dispositivo": "cpu", "segundos": seg, "parametros": json.dumps(kw), **metricas(S.loc[f.index, c], f[c])})
            print(f"h={h:2d} {nombre:16s} [cpu] {seg:6.1f} s", flush=True)
        for nombre in ("Ridge", "Random Forest", "XGBoost"):
            est, dev = MODELOS[nombre]
            t0 = time.time(); f = ml_origen_movil(P3, h, est, MALLAS_P3[nombre], CFG["P3_MESES_PRUEBA"]); seg = time.time() - t0
            f.to_parquet(PRED / f"p3_ajuste_fino_h{h}_{nombre}.parquet")
            for c in OBJ3: filas.append({"modelo": nombre, "h": h, "objetivo": c, "dispositivo": dev, "segundos": seg, "parametros": "", **metricas(S.loc[f.index, c], f[c])})
            print(f"h={h:2d} {nombre:16s} [{dev}] {seg:6.1f} s", flush=True)
    res = pd.DataFrame(filas); res.to_csv(REPORTS / "p3_ajuste_fino_metricas.csv", index=False); return res
if CORRER["p3"]: a3 = correr_p3_ajuste()
a3 = pd.read_csv(REPORTS / "p3_ajuste_fino_metricas.csv"); a3.pivot_table(index="modelo", columns="h", values="RMSE").round(3).sort_values(1)

## 5.4 Mejora frente a los modelos base

In [ ]:
for p in ("p1", "p2"):
    f = lambda r: r.groupby("modelo")["RMSE"].mean()
    b, a = pd.read_csv(REPORTS / f"{p}_base_metricas.csv"), pd.read_csv(REPORTS / f"{p}_ajuste_fino_metricas.csv")
    print(p); print(pd.DataFrame({"base": f(b), "ajuste_fino": f(a)}).assign(mejora_pct=lambda d: 100 * (1 - d.ajuste_fino / d.base)).round(3).sort_values("ajuste_fino"))
for p in ("p1", "p2"):
    a = pd.read_csv(REPORTS / f"{p}_ajuste_fino_metricas.csv").drop_duplicates("modelo")
    print(p); print(a[["modelo", "parametros", "segundos"]].to_string(index=False))